## SQuARE Threshold Sensitivity — Wikidata (Experiments A + B)

Implements `notebooks/evaluation-RQ1/THRESHOLD_SENSITIVITY.md`, the reviewer
defense for the SQuARE construction thresholds (positives: Jaccard > **0.8**,
negatives: Jaccard < **0.2**):

- **Experiment A** — pairwise answer-set Jaccard distribution over the full
  result-set corpus: shows 0.8 / 0.2 sit in the low-density valley of a
  bimodal distribution (grey-zone mass in [0.2, 0.8] reported).
- **Experiment B** — evaluation-side τ sweep: recompute RQ1 relevance labels
  at τ ∈ {0.6, 0.7, 0.8, 0.9} **without retraining** and show the
  SparqlCE-vs-baseline ranking gap is stable → the "threshold sensitivity
  mini-table".

Requires the Wikidata SparqlCE checkpoint and the answer-set cache
`datasets/result_sets_wikidata.json` (~5.9 GB, not in git — see
`datasets/README.md`). Loading converts each result row to a 64-bit hash,
so steady-state memory is small, but budget ~3x the file size in RAM for
the parse itself. Outputs go to `results/threshold_sensitivity/`.


In [ ]:
import os
import sys
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
# evaluate_rq1.py lives in notebooks/evaluation-RQ1/
_RQ1_DIR = _PROJECT_ROOT / "notebooks" / "evaluation-RQ1"
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

TESTSET_PATH  = _PROJECT_ROOT / "datasets" / "wikidata_testset_RQ1.json"
RESULTS_PATH  = _PROJECT_ROOT / "datasets" / "result_sets_wikidata.json"
SPARQLCE_CKPT = _PROJECT_ROOT / "sparqlce" / "models" / "wikidata_model_mrr_0.3238.pt"
OUT_DIR       = _PROJECT_ROOT / "results" / "threshold_sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TAUS   = [0.6, 0.7, 0.8, 0.9]   # 0.8 is the paper's construction threshold
TOPK   = 10                     # mini-table reports MRR@10 / Hits@10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# The Wikidata cache is ~5.9 GB; hashing each row to a 64-bit int keeps the
# in-memory sets small (Jaccard is unchanged up to negligible collisions).
with open(RESULTS_PATH, encoding="utf-8") as f:
    _raw = json.load(f)
result_sets = {}
for q, rows in _raw.items():
    if rows:
        result_sets[q] = frozenset(hash(r) for r in rows)
del _raw
print(f"queries with non-empty results: {len(result_sets)}")


def jaccard(a: frozenset, b: frozenset) -> float:
    union = len(a | b)
    return len(a & b) / union if union else 0.0


### Experiment A — pairwise Jaccard distribution

All unordered pairs over the queries with non-empty cached results
(~2.5M pairs for 2,229 queries; some Wikidata result sets are large, so
expect this to take longer than the DBpedia run).


In [ ]:
queries = list(result_sets)
sets    = [result_sets[q] for q in queries]

pair_j = np.empty(len(queries) * (len(queries) - 1) // 2, dtype=np.float32)
k = 0
for i in tqdm(range(len(sets)), desc="pairwise Jaccard"):
    Ri = sets[i]
    for j in range(i + 1, len(sets)):
        pair_j[k] = jaccard(Ri, sets[j])
        k += 1

grey  = float(np.mean((pair_j >= 0.2) & (pair_j <= 0.8)))
lo    = float(np.mean(pair_j < 0.2))
hi    = float(np.mean(pair_j > 0.8))
print(f"pairs: {len(pair_j):,}")
print(f"Jaccard < 0.2 : {lo:.2%}   (negative region)")
print(f"0.2–0.8 grey  : {grey:.2%}  <-- the number for the paper")
print(f"Jaccard > 0.8 : {hi:.2%}   (positive region)")

np.save(OUT_DIR / "pairwise_jaccard_wikidata.npy", pair_j)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(pair_j, bins=50, color="#4878a8", edgecolor="white", linewidth=0.3)
ax.set_yscale("log")
for x, style in [(0.2, "--"), (0.8, "--")]:
    ax.axvline(x, color="#d62728", linestyle=style, linewidth=1.2)
ax.text(0.5, 0.92, f"grey zone [0.2, 0.8]: {grey:.1%} of pairs",
        transform=ax.transAxes, ha="center", fontsize=10)
ax.set_xlabel("Answer-set Jaccard overlap")
ax.set_ylabel("Pair count (log)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "jaccard_histogram_wikidata.pdf", dpi=300, bbox_inches="tight")
print(f"saved {OUT_DIR / 'jaccard_histogram_wikidata.pdf'}")


### Experiment B — evaluation-side τ sweep (no retraining)

Same corpus construction as the `06_eval_*` notebooks (`evaluate_rq1.py`),
but relevance is recomputed directly from answer-set Jaccard at each τ:
candidate `c` is relevant to anchor `a` iff `J(R_a, R_c) ≥ τ`. Embeddings are
computed **once** per model; only the labels change across τ.


In [ ]:
from evaluate_rq1 import load_triplets, build_global_corpus, build_eval_items

test_data = load_triplets(str(TESTSET_PATH))
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)

# EvalItem is a tuple (anchor, pool, positives); anchors need an oracle
# result to define Jaccard relevance
def _res(q):
    return result_sets.get(q, result_sets.get(q.strip()))

anchors_ok = [it[0] for it in eval_items if _res(it[0])]
corpus_ok  = [c for c in global_corpus if _res(c)]
print(f"anchors: {len(eval_items)} -> with oracle results: {len(anchors_ok)}")
print(f"corpus:  {len(global_corpus)} -> with oracle results: {len(corpus_ok)}")

# Precompute the anchor x corpus Jaccard matrix (labels for every tau)
J = np.zeros((len(anchors_ok), len(corpus_ok)), dtype=np.float32)
for i, a in enumerate(tqdm(anchors_ok, desc="relevance Jaccard")):
    Ra = _res(a)
    for j, c in enumerate(corpus_ok):
        if c != a:
            J[i, j] = jaccard(Ra, _res(c))


In [ ]:
# --- Embed anchors + corpus once per model ---
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from sparqlce.embedder import SPARQLEmbedder

def embed_texts(encode_fn, texts, batch=32):
    out = [encode_fn(texts[i:i + batch]) for i in range(0, len(texts), batch)]
    return normalize(np.vstack(out))

model_sims = {}   # name -> [n_anchors, n_corpus] cosine matrix

sparqlce = SPARQLEmbedder(str(SPARQLCE_CKPT))
A = embed_texts(sparqlce.encode, anchors_ok)
C = embed_texts(sparqlce.encode, corpus_ok)
model_sims["SparqlCE"] = A @ C.T
del sparqlce

for display, model_id, prefix in [
    ("E5-base-v2", "intfloat/e5-base-v2", "query: "),
    ("Qwen3-Embedding", "Qwen/Qwen3-Embedding-0.6B", ""),
]:
    st = SentenceTransformer(model_id, device=DEVICE)
    A = st.encode([prefix + q for q in anchors_ok], convert_to_numpy=True,
                  normalize_embeddings=True, show_progress_bar=True, batch_size=8)
    C = st.encode([prefix + q for q in corpus_ok], convert_to_numpy=True,
                  normalize_embeddings=True, show_progress_bar=True, batch_size=8)
    model_sims[display] = A @ C.T
    del st
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print({k: v.shape for k, v in model_sims.items()})


In [ ]:
# --- Rank once per model, relabel per tau ---
def mrr_hits_at_k(sims, J, tau, k, anchors, corpus):
    """MRR@k and Hits@k where relevance := J >= tau (anchor itself excluded)."""
    rrs, hits = [], []
    order = np.argsort(-sims, axis=1)
    for i in range(sims.shape[0]):
        rel = J[i] >= tau
        if not rel.any():
            continue                     # anchor has no relevant candidate at this tau
        rr, hit = 0.0, 0.0
        rank = 0
        for j in order[i]:
            if corpus[j] == anchors[i]:
                continue                 # skip self
            rank += 1
            if rank > k:
                break
            if rel[j]:
                rr, hit = 1.0 / rank, 1.0
                break
        rrs.append(rr); hits.append(hit)
    return float(np.mean(rrs)), float(np.mean(hits)), len(rrs)


rows = []
for tau in TAUS:
    for name, sims in model_sims.items():
        mrr, h, n = mrr_hits_at_k(sims, J, tau, TOPK, anchors_ok, corpus_ok)
        rows.append({"tau": tau, "model": name,
                     f"MRR@{TOPK}": mrr, f"Hits@{TOPK}": h, "n_anchors": n})

mini = pd.DataFrame(rows)
pivot = mini.pivot(index="tau", columns="model", values=f"MRR@{TOPK}").round(4)
pivot["gap (SparqlCE / best baseline)"] = (
    pivot["SparqlCE"] / pivot.drop(columns="SparqlCE").max(axis=1)).round(2)
print(pivot)

mini.to_json(OUT_DIR / "tau_sweep_wikidata.json", orient="records", indent=2)

print("\n% LaTeX mini-table rows (MRR@10 per tau):")
for tau in TAUS:
    r = pivot.loc[tau]
    marker = r"\;(paper)" if tau == 0.8 else ""
    print(f"$\\tau={tau}${marker} & " +
          " & ".join(f"{r[m]:.4f}" for m in model_sims) +
          f" & {r['gap (SparqlCE / best baseline)']:.1f}$\\times$ \\\\")


### What to put in the paper

- **Experiment A:** one sentence next to the SQuARE construction paragraph:
  *"the pairwise answer-set Jaccard distribution is strongly bimodal — only
  X% of pairs fall in the grey zone [0.2, 0.8] — so the labels are insensitive
  to the exact cut"*, optionally with the histogram as an appendix figure.
- **Experiment B:** the printed mini-table (τ ∈ {0.6,…,0.9} × models, MRR@10
  and the SparqlCE-vs-best-baseline ratio). Expected story: absolute numbers
  shift with τ, the SparqlCE-vs-baseline gap does not. On Wikidata the
  grey zone should be even smaller than DBpedia's (~3% of annotated pairs).
- Experiment C (construction-side retrain at Jaccard > {0.7, 0.9}) is
  specified in `THRESHOLD_SENSITIVITY.md` — hold for the rebuttal.
